# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alien-is-here/FlyRank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Answer:**

**Finding 1 — Health Score feature importance**

The paper reports that Average Position (43%), Impressions (32%), Scroll Depth (15%), and CTR (8%) dominate the Random Forest's prediction of Health Score.

Methodology question:
The Health Score itself is constructed from Impressions, Position, CTR, and Scroll Depth. Therefore, the model is partly predicting a label using the same variables that created that label. Could the finding be interpreted more narrowly as descriptive model behavior, rather than evidence that these features independently cause better SEO performance?

**Finding 2 — 71% Logistic Regression accuracy**

The paper reports 71% holdout accuracy for Logistic Regression separating growing from declining pages.

Methodology question:
The paper says the ML pipeline uses an 80/20 split, but because pages can share client-level patterns, would a client-grouped or time-aware validation provide a stronger test of whether the result generalizes to unseen clients or future periods?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Answer:**

The Random Forest achieved a Precision@50 of 0.74 under the Week-5 random split, compared with 0.62 under the grouped-by-client split. The decrease suggests that the random-split result may have been optimistic when estimating performance on unseen clients. Under the grouped split, 31 of the top 50 ranked pages were observed declines.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
feature_cols = [
    "ctr",
    "avg_position",
    "engagement_rate"
]

X = df[feature_cols]

y = (df["trend_direction"] == "down").astype(int)

groups = df[["client_id","content_id"]]

In [4]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Group pages by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

# Train the same Random Forest
rf_group = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_group.fit(X_train_group, y_train_group)

# Rank test pages by probability of decline
y_prob_group = rf_group.predict_proba(X_test_group)[:, 1]

ranking = pd.DataFrame({
    "actual": y_test_group.values,
    "score": y_prob_group
})

ranking = ranking.sort_values(
    "score",
    ascending=False
)

# Top 50 pages
top_50 = ranking.head(50)

precision_at_50_after = top_50["actual"].mean()

print("Before Precision@50:", 0.740)
print("After Precision@50:", precision_at_50_after)
print("True declines in top 50:", int(top_50["actual"].sum()))

Before Precision@50: 0.74
After Precision@50: 0.62
True declines in top 50: 31


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Answer:**

I audited the final feature set against the leakage risks identified in Week 3.

The target is created from `trend_direction`, so target-derived fields such as `trend_direction`, `trend_pct`, and `is_declining_label` must not be used as features. Identifiers such as `client_id` and `content_id` are also excluded from the model features.

The final feature set contains only `ctr`, `avg_position`, and `engagement_rate`, so there is no direct feature-name overlap with the known leakage columns.

However, this column-level check cannot prove that the feature windows are completely separated from the target window. Therefore, possible temporal overlap between the performance features and the target definition remains a limitation that should be verified from the underlying data timing.

In [5]:
leakage_cols = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "client_id",
    "content_id"
}

# Check final model features
feature_leaks = set(feature_cols) & leakage_cols

print("Final features:", feature_cols)
print("Known leakage columns:", sorted(leakage_cols))
print("Feature overlap:", feature_leaks)

assert len(feature_leaks) == 0, "Potential leakage found in feature set!"

print("No direct leakage columns found in the final feature set.")

Final features: ['ctr', 'avg_position', 'engagement_rate']
Known leakage columns: ['client_id', 'content_id', 'is_declining_label', 'trend_direction', 'trend_pct']
Feature overlap: set()
No direct leakage columns found in the final feature set.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Answer:**


In the evaluated data, the Random Forest showed useful directional performance for prioritizing potentially declining pages for review. It achieved a measured Precision@50 of 0.74 with the Week-5 random split and 0.62 with the grouped-by-client split. These results suggest that the model can serve as decision-support for prioritizing pages for human review, but they do not establish that the model will generalize equally well to unseen clients or that refreshing a page will cause recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.